# AI-RAG · Week 11 — Notebook 1
## The Four Metrics & RAGAS

*The measure-it week, made runnable.* This notebook walks the four RAG-specific metrics — **faithfulness, answer relevance, context precision, context recall** — the golden set they score against, the row-builder that feeds them, and the task-success judge. Every cell runs **offline** (deterministic token-overlap *proxies*, no API key, no cost) so you can run it top-to-bottom while you explain it. Set an `OPENAI_API_KEY` and remove the `EVAL_OFFLINE` line in setup to switch to **live RAGAS**.

> The code below is lifted **verbatim** from the package modules (`src/eval/*.py`). The only changes are notebook glue (paths, and dropping the relative-import lines because the code is defined inline here).

### Setup

In [1]:
# --- notebook setup (delivery glue only) -------------------------------------
# 1) ensure pydantic is available   2) write the golden set locally
# 3) force the OFFLINE proxy path so this runs top-to-bottom with no API key/cost.
try:
    import pydantic  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pydantic"])

import os
from pathlib import Path
os.environ["EVAL_OFFLINE"] = "1"   # <- set an OPENAI_API_KEY and remove this line to run LIVE RAGAS

GOLDEN_JSONL = r"""{"id": "g001", "question": "What is the company's leave policy?", "ideal_answer": "Employees get 20 days of paid leave per year, accrued monthly at 1.67 days per month. Unused leave caps at 30 days; anything above that lapses on 31 December.", "notes": "happy-path. Source: HR-policy.pdf §3.1."}
{"id": "g002", "question": "How do I request a reimbursement for a work-related expense under $100?", "ideal_answer": "Submit the receipt in the expense portal within 30 days of the purchase. Items under $100 are auto-approved if they fit a standard category (travel, meals, supplies). Submission outside 30 days requires manager approval.", "notes": "happy-path. Source: Finance-policy.pdf §5.4."}
{"id": "g003", "question": "What is the standard working week?", "ideal_answer": "The standard working week is 40 hours, Monday to Friday, 09:00 to 17:30 with a one-hour lunch. Flexible start times between 08:00 and 10:00 are permitted with manager approval.", "notes": "happy-path. Source: HR-policy.pdf §1.2."}
{"id": "g004", "question": "What's the policy on bringing personal devices to work (BYOD)?", "ideal_answer": "Personal devices may be used to access company email and approved SaaS apps via the MDM enrollment. Devices must be password-protected, encrypted, and patched within 30 days of OS updates. Local storage of company documents is prohibited.", "notes": "happy-path. Source: IT-security.pdf §2.3."}
{"id": "g005", "question": "How do I request parental leave?", "ideal_answer": "Submit a parental leave request via the HR portal at least 8 weeks before the expected leave start date. Primary caregivers are entitled to 16 weeks paid; secondary caregivers to 4 weeks paid. Additional unpaid leave up to 12 months may be granted with manager approval.", "notes": "happy-path. Source: HR-policy.pdf §3.4."}
{"id": "g006", "question": "What's the dress code for client meetings?", "ideal_answer": "Business smart for in-person client meetings: collared shirt or blouse, smart trousers or skirt, closed-toe shoes. Jeans and sneakers are acceptable for internal meetings but not for client-facing ones.", "notes": "happy-path. Source: HR-policy.pdf §2.1."}
{"id": "g007", "question": "How do I report a security incident?", "ideal_answer": "Email security@company.com immediately and create a ticket in the IT portal flagged as 'Security Incident'. For incidents involving customer data, also notify the Data Protection Officer within 24 hours. Do not investigate the incident yourself — preserve evidence.", "notes": "happy-path. Source: IT-security.pdf §4.1."}
{"id": "g008", "question": "What's the process for requesting new software?", "ideal_answer": "Software requests go through the IT portal. Pre-approved software (listed in the IT catalogue) is auto-installed within 1 business day. New requests require justification, a security review (2-5 business days), and budget approval if the cost exceeds $500/year.", "notes": "happy-path. Source: IT-policy.pdf §3.2."}
{"id": "g009", "question": "Who do I contact about my pay or benefits?", "ideal_answer": "For pay queries, contact Payroll at payroll@company.com or via the HR portal under 'Pay & Benefits'. For benefits (health insurance, retirement, etc.), contact HR Benefits at benefits@company.com.", "notes": "happy-path. Source: HR-policy.pdf §6."}
{"id": "g010", "question": "What's the policy on working from home?", "ideal_answer": "Employees may work remotely up to 3 days per week with manager approval. Remote work requires a stable internet connection, a quiet workspace, and availability during core hours (10:00-15:00 local time). Full-time remote arrangements require VP-level approval.", "notes": "happy-path. Source: HR-policy.pdf §1.5."}
{"id": "g011", "question": "How do I book a meeting room?", "ideal_answer": "Meeting rooms are booked via the Outlook calendar. Rooms over 12 people require booking at least 24 hours in advance. Recurring meetings should not block premium rooms (those with video conferencing) for more than 2 hours per week.", "notes": "happy-path. Source: Facilities-policy.pdf §1.1."}
{"id": "g012", "question": "What's the company's policy on accepting gifts from clients?", "ideal_answer": "Gifts under $50 in value may be accepted with line-manager notification. Gifts above $50 must be declined or, if culturally inappropriate to decline, donated to charity with HR notified. Cash, gift cards, and personal services may never be accepted.", "notes": "happy-path. Source: Compliance-policy.pdf §2.1."}
{"id": "g013", "question": "How do I onboard a new contractor?", "ideal_answer": "Submit a contractor onboarding request via the HR portal at least 5 business days before the start date. Include scope of work, contract duration, budget code, and the contractor's NDA status. IT provisioning takes 2 business days from approval.", "notes": "happy-path. Source: HR-policy.pdf §5.1."}
{"id": "g014", "question": "What training is required for new employees?", "ideal_answer": "All new employees complete three mandatory trainings within 30 days of joining: (1) Information Security & Data Protection, (2) Code of Conduct & Anti-Harassment, (3) Health & Safety. Tracked via the L&D portal; manager is notified if not completed by day 30.", "notes": "happy-path. Source: HR-policy.pdf §4.2."}
{"id": "g015", "question": "How does the remote work policy interact with the requirement to attend in-person team meetings?", "ideal_answer": "Remote work allowance (up to 3 days per week) is subject to attending mandatory in-person team meetings as scheduled by the manager. Teams typically schedule one anchor in-person day per week. Recurring conflicts should be resolved by the team's manager, who may approve exceptions on a case-by-case basis.", "notes": "harder · multi-hop. Cross-references HR-policy §1.5 (remote) AND §2.2 (team meetings). Tests integration across two source sections."}
{"id": "g016", "question": "If I'm on parental leave and a serious security incident happens in my product area, am I expected to respond?", "ideal_answer": "No. Employees on parental leave are not expected to respond to work matters, including security incidents. The on-call rotation and the team's secondary contact (named in each team's runbook) cover incident response during any team member's absence. If the matter is critical and only the absent employee has the knowledge, the line manager escalates to the VP for coverage.", "notes": "harder · multi-hop. Cross-references HR-policy §3.4 (parental leave) AND IT-security §4.1 (incident response). Tests respect for leave + named escalation path."}
{"id": "g017", "question": "What's the right channel to report a potential conflict of interest involving a vendor my partner works for?", "ideal_answer": "Disclose the potential conflict via the Compliance portal under 'Conflict of Interest Disclosure'. Include the vendor name, your partner's role, and any business relationship you have with the vendor. The Compliance team reviews disclosures within 5 business days and may require recusal from procurement or vendor-management decisions involving that vendor.", "notes": "harder. Combines Compliance-policy §3 (disclosures) and process detail. Tests nuanced conflict-of-interest reporting."}
{"id": "g018", "question": "Can I expense a Coursera certification that's not in the L&D catalogue if it directly supports my project?", "ideal_answer": "Yes, with manager pre-approval. Submit a request via the L&D portal under 'Out-of-catalogue learning' with a one-paragraph business justification. Certifications up to $1,000 per year per employee are typically approved for project-relevant skills. Reimbursement requires successful completion (certificate issued) within 6 months.", "notes": "harder. Cross-references L&D-policy §2.4 and Finance §5. Tests handling of off-catalogue requests with conditions."}
{"id": "g019", "question": "What is the company's stance on cryptocurrency investing as part of the benefits package?", "ideal_answer": "Cryptocurrency investing is not part of the benefits package. The answer is not in the company's policy documents. The assistant should say so clearly and suggest the employee contact HR for clarification.", "notes": "edge. Tests graceful refusal — answer NOT in docs. Should not hallucinate."}
{"id": "g020", "question": "Will the company sponsor my visa for a personal trip to a country requiring a business invitation letter?", "ideal_answer": "Visa support is only provided for company-sanctioned business travel — see Travel-policy §3.2. Personal travel visas are the employee's responsibility. The company does not issue invitation letters for personal trips.", "notes": "edge. Tests boundary-setting — adjacent to a real policy but clearly outside its scope. Should redirect, not invent."}"""
Path("golden_set").mkdir(exist_ok=True)
Path("golden_set/capstone_v1.jsonl").write_text(GOLDEN_JSONL + "\n")
print("golden set written:", len(GOLDEN_JSONL.splitlines()), "entries · EVAL_OFFLINE =", os.environ["EVAL_OFFLINE"])


golden set written: 20 entries · EVAL_OFFLINE = 1


## 1 · The golden set — `src/eval/golden.py`

The golden set is the ground truth every later week's KPI is computed against — 20 entries, ids `g001..g020`, each with a `question`, an `ideal_answer` (RAGAS's `ground_truth`), and `notes` (category + source). Treat it as infrastructure, not a fixture.

In [2]:
"""src/eval/golden.py — loader and validator for the capstone golden set.

Lineage: carried forward unchanged from W5. The golden set is the ground truth
that every later week's KPI is computed against — treat it as infrastructure,
not a one-off fixture. As of W10 it lives at `golden_set/capstone_v1.jsonl`
(the W5 set plus W7/W9 additions), 20 entries, ids g001..g020.

A golden entry has four fields:
  • id            — stable identifier (g001..g020). Referenced in eval runs
                    and across docs.
  • question      — exactly as a real user would phrase it.
  • ideal_answer  — full reference text. RAGAS uses this as `ground_truth`;
                    the judge uses it as the ideal answer. For open-ended
                    questions this may be a properties dict.
  • notes         — category (happy/harder/edge), source doc, why it matters.
"""
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any

from pydantic import BaseModel, Field, field_validator

_ID_PATTERN = re.compile(r"^g\d{3}$")


class GoldenEntry(BaseModel):
    """One row of the golden set."""

    id: str = Field(..., description="Stable identifier g001..g999")
    question: str = Field(..., min_length=1)
    ideal_answer: str | dict[str, Any] = Field(...)
    notes: str = Field(default="", description="Category, source, why this matters")

    @field_validator("id")
    @classmethod
    def _id_shape(cls, v: str) -> str:
        if not _ID_PATTERN.match(v):
            raise ValueError(f"id must match g001..g999; got {v!r}")
        return v

    @property
    def ground_truth(self) -> str:
        """RAGAS expects a string ground_truth. Flatten dict ideals to text."""
        if isinstance(self.ideal_answer, dict):
            return json.dumps(self.ideal_answer)
        return self.ideal_answer


def load_golden_set(path: str | Path) -> list[GoldenEntry]:
    """Read a .jsonl file. Each non-blank, non-comment line is one entry."""
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"Golden set not found at {p}")
    entries: list[GoldenEntry] = []
    with p.open() as f:
        for lineno, raw in enumerate(f, start=1):
            line = raw.strip()
            if not line or line.startswith("//"):
                continue
            try:
                data = json.loads(line)
            except json.JSONDecodeError as e:
                raise ValueError(f"{p}:{lineno} — malformed JSON: {e.msg}") from e
            entries.append(GoldenEntry.model_validate(data))
    return entries


def validate_golden_set(path: str | Path) -> dict[str, Any]:
    """Validate the file. Returns a summary dict; raises on errors."""
    entries = load_golden_set(path)

    ids = [e.id for e in entries]
    if len(ids) != len(set(ids)):
        seen: set[str] = set()
        dupes = [i for i in ids if i in seen or seen.add(i)]  # noqa: B015
        raise ValueError(f"Duplicate ids in {path}: {sorted(set(dupes))}")

    def _bucket(note: str) -> str:
        low = note.lower()
        if low.startswith("happy"):
            return "happy"
        if low.startswith("harder") or low.startswith("multi"):
            return "harder"
        if low.startswith("edge"):
            return "edge"
        return "uncategorised"

    counts: dict[str, int] = {}
    for e in entries:
        counts[_bucket(e.notes)] = counts.get(_bucket(e.notes), 0) + 1

    return {"path": str(path), "n": len(entries), "ids": ids, "coverage": counts}


**Load and validate it:**

In [3]:
golden = load_golden_set('golden_set/capstone_v1.jsonl')
summary = validate_golden_set('golden_set/capstone_v1.jsonl')
print('entries:', summary['n'])
print('coverage:', summary['coverage'])
print('first:', golden[0].id, '—', golden[0].question)

entries: 20
coverage: {'happy': 14, 'harder': 4, 'edge': 2}
first: g001 — What is the company's leave policy?


## 2 · The four metrics, two modes — `src/eval/ragas_metrics.py`

Each metric takes a list of **rows** — `{id, question, answer, contexts, ground_truth, ...}` — and returns a `MetricResult` (per-row scores + an aggregate). **Live** = real RAGAS, one LLM-judge call per row — the numbers you defend at M2. **Offline** = deterministic token-overlap *proxies*, labelled `offline-proxy` so a proxy is never mistaken for a real RAGAS score. The heavy `ragas`/`datasets` imports are **lazy** (inside the live functions), so the offline path needs nothing installed. (We define the functions here, then score real rows once we've built them in section 3.)

In [4]:
"""src/eval/ragas_metrics.py — the four RAG-specific metrics, two ways to run.

W11 standardises on RAGAS for the four RAG metrics:
  • faithfulness        — is the answer supported by the retrieved context?
  • answer_relevance    — does the answer address the question?
  • context_precision   — are the top-K retrieved chunks the right ones?
  • context_recall      — did we retrieve everything we needed?

Each public function takes a list of `rows` and returns a `MetricResult`
(per-row scores + an aggregate). A row is a dict with:
    {"id", "question", "answer", "contexts": [str, ...], "ground_truth"}

TWO MODES
---------
LIVE  (default when an API key is set and `ragas` is importable):
      real RAGAS metrics, each backed by an LLM judge call per row. This is
      what produces the numbers you put in the snapshot and defend at M2.

OFFLINE (set EVAL_OFFLINE=1, or when `ragas` isn't installed):
      deterministic token-overlap *proxies*. These are NOT RAGAS numbers —
      they exist so the eval suite runs in CI / on a laptop with no API key,
      proving the wiring is sound. Every offline MetricResult is labelled
      `mode="offline-proxy"` so a proxy number is never mistaken for a real
      RAGAS score. The smoke test runs this path.

This split is the whole point of "eval as code": the suite is runnable and
assertable on every commit (offline), and produces real numbers on demand
(live). See docs/eval/README.md.
"""
from __future__ import annotations

import os
import re
from dataclasses import dataclass, field


# ─── Result container ────────────────────────────────────────────────────────
@dataclass
class MetricResult:
    """One metric's outcome across all rows."""

    name: str
    mode: str                       # "live" or "offline-proxy"
    aggregate: float                # mean across rows, 0..1
    per_row: dict[str, float] = field(default_factory=dict)  # id -> score

    def __str__(self) -> str:
        return f"{self.name}={self.aggregate:.3f} ({self.mode}, n={len(self.per_row)})"


# ─── Mode detection ──────────────────────────────────────────────────────────
def offline_mode() -> bool:
    """Offline if explicitly forced, or if ragas / an API key is unavailable."""
    if os.getenv("EVAL_OFFLINE", "").strip() in {"1", "true", "TRUE", "yes"}:
        return True
    if not os.getenv("OPENAI_API_KEY"):
        return True
    try:
        import ragas  # noqa: F401
    except Exception:
        return True
    return False


# ─── Token helpers (offline proxy) ───────────────────────────────────────────
_STOP = {
    "the", "a", "an", "and", "or", "but", "if", "of", "to", "in", "on", "for",
    "with", "is", "are", "was", "were", "be", "been", "as", "at", "by", "it",
    "this", "that", "these", "those", "from", "your", "you", "may", "must",
    "can", "do", "does", "how", "what", "who", "i", "my", "me", "we", "our",
    "per", "up", "any", "all", "via", "not", "no", "they", "their",
}


def _tokens(text: str) -> set[str]:
    if not text:
        return set()
    raw = re.findall(r"[a-z0-9]+", text.lower())
    return {t for t in raw if len(t) > 2 and t not in _STOP}


def _overlap_fraction(a: set[str], b: set[str]) -> float:
    """Fraction of `a`'s tokens that appear in `b`. 0..1. Empty `a` -> 0.0."""
    if not a:
        return 0.0
    return len(a & b) / len(a)


# ─── Offline proxies (deterministic, dependency-free) ────────────────────────
def _proxy(name: str, rows: list[dict], fn) -> MetricResult:
    per_row = {r["id"]: round(fn(r), 4) for r in rows}
    agg = round(sum(per_row.values()) / len(per_row), 4) if per_row else 0.0
    return MetricResult(name=name, mode="offline-proxy", aggregate=agg, per_row=per_row)


def _faithfulness_proxy(r: dict) -> float:
    # How much of the answer is covered by the retrieved context?
    ans = _tokens(r.get("answer", ""))
    ctx = _tokens(" ".join(r.get("contexts", [])))
    return _overlap_fraction(ans, ctx)


def _answer_relevance_proxy(r: dict) -> float:
    # Does the answer address the question? Blend (a) how many of the question's
    # key terms the answer covers with (b) whether the answer engages the topic
    # at all. Short questions make pure token-overlap jumpy, so the topical-touch
    # term stabilises a correct answer near ~0.7 while a truly off-topic answer
    # (no shared terms) still collapses to 0.0.
    q = _tokens(r.get("question", ""))
    ans = _tokens(r.get("answer", ""))
    if not q:
        return 0.0
    coverage = _overlap_fraction(q, ans)            # 0..1
    touches = 1.0 if (q & ans) else 0.0             # any shared key term?
    return round(0.5 * coverage + 0.5 * touches, 4)


def _context_precision_proxy(r: dict) -> float:
    # Of the retrieved chunks, what fraction are relevant to the ground truth?
    gt = _tokens(r.get("ground_truth", ""))
    contexts = r.get("contexts", []) or []
    if not contexts:
        return 0.0
    relevant = sum(1 for c in contexts if _tokens(c) & gt)
    return relevant / len(contexts)


def _context_recall_proxy(r: dict) -> float:
    # How much of the ground truth is present across the retrieved chunks?
    gt = _tokens(r.get("ground_truth", ""))
    ctx = _tokens(" ".join(r.get("contexts", [])))
    return _overlap_fraction(gt, ctx)


# ─── Live RAGAS ──────────────────────────────────────────────────────────────
def _ragas_score(name: str, rows: list[dict], metric, column: str) -> MetricResult:
    """Run one RAGAS metric over the rows. Imported lazily (heavy deps).

    Targets ragas ~0.1.x (see requirements.txt for the pin). RAGAS has changed
    column names and metric imports across versions — if this raises, check the
    pinned version and the Lab Guide's 'RAGAS version pinning' note.
    """
    from datasets import Dataset
    from ragas import evaluate

    ds = Dataset.from_dict({
        "question":     [r["question"] for r in rows],
        "answer":       [r["answer"] for r in rows],
        "contexts":     [list(r.get("contexts", [])) for r in rows],
        "ground_truth": [r.get("ground_truth", "") for r in rows],
    })
    result = evaluate(ds, metrics=[metric])
    df = result.to_pandas()

    per_row = {
        rows[i]["id"]: round(float(df[column].iloc[i]), 4)
        for i in range(len(rows))
    }
    agg = round(float(df[column].mean()), 4)
    return MetricResult(name=name, mode="live", aggregate=agg, per_row=per_row)


# ─── Public API — one function per metric ────────────────────────────────────
def score_faithfulness(rows: list[dict]) -> MetricResult:
    if offline_mode():
        return _proxy("faithfulness", rows, _faithfulness_proxy)
    from ragas.metrics import faithfulness
    return _ragas_score("faithfulness", rows, faithfulness, "faithfulness")


def score_answer_relevance(rows: list[dict]) -> MetricResult:
    if offline_mode():
        return _proxy("answer_relevance", rows, _answer_relevance_proxy)
    from ragas.metrics import answer_relevancy
    return _ragas_score("answer_relevance", rows, answer_relevancy, "answer_relevancy")


def score_context_precision(rows: list[dict]) -> MetricResult:
    if offline_mode():
        return _proxy("context_precision", rows, _context_precision_proxy)
    from ragas.metrics import context_precision
    return _ragas_score("context_precision", rows, context_precision, "context_precision")


def score_context_recall(rows: list[dict]) -> MetricResult:
    if offline_mode():
        return _proxy("context_recall", rows, _context_recall_proxy)
    from ragas.metrics import context_recall
    return _ragas_score("context_recall", rows, context_recall, "context_recall")


# Convenience: all four at once (used by run_eval.py and the baseline doc).
ALL_METRICS = (
    score_faithfulness,
    score_answer_relevance,
    score_context_precision,
    score_context_recall,
)


def score_all(rows: list[dict]) -> dict[str, MetricResult]:
    results = [fn(rows) for fn in ALL_METRICS]   # one call each (matters live!)
    return {r.name: r for r in results}


## 3 · Build the rows — `src/eval/harness.py`

The harness builds the rows one way offline (a deterministic synthetic pipeline) and another way live (your real capstone). A few rows are **deliberately degraded** so the report has real signal: `g019` hallucinates, `g015`/`g005` suffer retrieval misses.

In [5]:
"""src/eval/harness.py — build the eval rows once, two ways.

Shared by eval/conftest.py (the pytest suite) and scripts/run_eval.py (the CLI),
so the offline synthetic pipeline and the live capstone pipeline are defined in
exactly one place.

Row shape (identical offline and live):
    {"id", "question", "answer", "contexts": [str,...], "ground_truth",
     "cache_hit", "latency_ms", "cost_usd", "notes"}
"""
from __future__ import annotations

from pathlib import Path

# GoldenEntry, load_golden_set, offline_mode are defined in the cells above

REPO_ROOT = Path.cwd()   # notebook: resolve paths from the working directory
GOLDEN_PATH = REPO_ROOT / "golden_set" / "capstone_v1.jsonl"

# KB version active at eval time. Live mode reads it from the pipeline; offline
# pins the W10 value so every eval run and snapshot is tagged by KB state (D4).
OFFLINE_KB_VERSION = "v2.w10"

# A clean distractor sharing almost no content tokens with policy answers, so
# context precision reflects relevant-vs-retrieved honestly.
_DISTRACTOR = (
    "The office kitchen is restocked every Monday morning and the building "
    "reception desk is open from eight until six."
)


def _split_two(text: str) -> tuple[str, str]:
    words = text.split()
    mid = max(1, len(words) // 2)
    return " ".join(words[:mid]), " ".join(words[mid:])


def _synth_row(entry: GoldenEntry, index: int) -> dict:
    """Deterministically build one (answer, contexts) pair for a golden entry.

    Most rows are faithful and complete. A few are deliberately degraded so the
    report has real signal: g019 hallucinates, g015/g005 suffer retrieval misses.
    """
    gt = entry.ground_truth
    c1, c2 = _split_two(gt)
    contexts = [c1, c2, _DISTRACTOR]   # 2 relevant + 1 distractor
    answer = gt

    if entry.id == "g019":
        answer = gt + (" The company also matches up to five percent of salary "
                       "into a managed crypto retirement fund.")   # hallucination
    elif entry.id == "g015":
        contexts = [c1, _DISTRACTOR]                                # multi-hop miss
    elif entry.id == "g005":
        contexts = [c1[: max(1, len(c1) // 2)], _DISTRACTOR]        # weak retrieval

    cache_hit = index < 13          # ~65% warm hit rate, matching wk10
    return {
        "id":           entry.id,
        "question":     entry.question,
        "answer":       answer,
        "contexts":     contexts,
        "ground_truth": gt,
        "cache_hit":    cache_hit,
        "latency_ms":   184 if cache_hit else 785,
        "cost_usd":     0.000041 if cache_hit else 0.000412,
        "notes":        entry.notes,
    }


def _live_rows(golden: list[GoldenEntry]) -> list[dict]:  # pragma: no cover - live only
    """Run the real capstone over the golden set. Needs the learner's src/rag."""
    try:
        from src.rag.qdrant_rag import ask_rag
        from src.rag.qdrant_store import QdrantStore
        from src.rag.retrieval import (
            build_bm25_index,
            hybrid_retrieve,
            load_chunks_from_qdrant,
        )
    except Exception as exc:
        raise RuntimeError(
            "Live eval needs your capstone code (src/rag/*) on the path and "
            "OPENAI_API_KEY + QDRANT_URL set. Run offline with EVAL_OFFLINE=1. "
            f"Import error: {exc}"
        ) from exc

    store = QdrantStore()
    chunks = load_chunks_from_qdrant(store)          # tombstone-filtered (W10)
    bm25 = build_bm25_index(chunks)
    rows: list[dict] = []
    for e in golden:
        res = ask_rag(e.question, store, k=3)
        # Full chunk text for RAGAS contexts comes from retrieval, not from
        # ask_rag's stripped citations (D3).
        retrieved = hybrid_retrieve(store, bm25, chunks, e.question, top_k=3)
        rows.append({
            "id":           e.id,
            "question":     e.question,
            "answer":       res["answer"],
            "contexts":     [c.get("text", "") for c in retrieved],
            "ground_truth": e.ground_truth,
            "cache_hit":    res.get("cache_hit", False),
            "latency_ms":   res.get("latency_ms", 0),
            "cost_usd":     res.get("cost_usd", 0.0),
            "notes":        e.notes,
        })
    return rows


def active_kb_version() -> str:
    if offline_mode():
        return OFFLINE_KB_VERSION
    try:  # pragma: no cover - live only
        from src.rag.pipeline import KB_VERSION
        return KB_VERSION
    except Exception:  # pragma: no cover
        return "unknown"


def load_golden() -> list[GoldenEntry]:
    return load_golden_set(GOLDEN_PATH)


def build_rows(golden: list[GoldenEntry] | None = None) -> list[dict]:
    """The (question, answer, contexts, ground_truth, ...) rows under test."""
    golden = golden or load_golden()
    if offline_mode():
        return [_synth_row(e, i) for i, e in enumerate(golden)]
    return _live_rows(golden)


**Build the rows and inspect one — plus the seeded failures:**

In [6]:
rows = build_rows()
print('rows:', len(rows), '| kb_version:', active_kb_version())

sample = rows[0]
print('\n-- a healthy row --')
print('id:', sample['id'])
print('answer:', sample['answer'][:90], '...')
print('contexts:', len(sample['contexts']), 'chunks')

print('\n-- the seeded failures --')
for r in rows:
    if r['id'] in ('g005', 'g015', 'g019'):
        print(r['id'], '| contexts:', len(r['contexts']), '| note:', r['notes'][:40])

rows: 20 | kb_version: v2.w10

-- a healthy row --
id: g001
answer: Employees get 20 days of paid leave per year, accrued monthly at 1.67 days per month. Unus ...
contexts: 3 chunks

-- the seeded failures --
g005 | contexts: 2 | note: happy-path. Source: HR-policy.pdf §3.4.
g015 | contexts: 2 | note: harder · multi-hop. Cross-references HR-
g019 | contexts: 3 | note: edge. Tests graceful refusal — answer NO


## 4 · Score the golden set

Now that we have both the metric functions and the rows, score them. **Faithfulness across the 20 questions** — then read the lowest rows (the seeded dips should surface):

In [7]:
faith = score_faithfulness(rows)
print(faith)   # name=..., mode=offline-proxy, aggregate, n

print('\nlowest-scoring rows:')
for rid, sc in sorted(faith.per_row.items(), key=lambda kv: kv[1])[:5]:
    print(f'  {rid}: {sc:.3f}')

faithfulness=0.925 (offline-proxy, n=20)

lowest-scoring rows:
  g005: 0.318
  g015: 0.556
  g019: 0.630
  g001: 1.000
  g002: 1.000


**All four at once** (`score_all` runs each metric exactly once — matters live!):

In [8]:
results = score_all(rows)
for name, res in results.items():
    print(f'{name:20s} {res.aggregate:.3f}   ({res.mode})')

faithfulness         0.925   (offline-proxy)
answer_relevance     0.731   (offline-proxy)
context_precision    0.667   (offline-proxy)
context_recall       0.944   (offline-proxy)


## 5 · The task-success judge — `src/eval/judge.py`

From W5, carried forward. RAGAS now owns the four RAG metrics; this LLM-as-judge is repurposed as the **task-success grader for KPI #1** — *did the answer meet the user's goal end-to-end?* — on a 3-dimension rubric (accuracy / groundedness / format, 1–4 each). It's **live-only** (needs an API key), so below we show the code and exercise its scoring logic locally **without** an API call.

In [9]:
"""src/eval/judge.py — LLM-as-judge (task-success grader).

Lineage: this is the W5 judge, carried forward into the capstone and lightly
hardened for W11. In W5 it was the headline eval tool; from W11 onwards RAGAS
owns the four RAG-specific metrics (faithfulness, answer relevance, context
precision, context recall) and this judge is repurposed as the *task-success*
grader for KPI #1 — "did the answer actually meet the user's goal end-to-end".

Two changes vs the W5 reference:
  1. Lazy client — we never instantiate AsyncOpenAI at import time, so the
     module imports cleanly in pytest with no API key set (W7/W8 footgun).
  2. A thin sync wrapper `judge_one_sync` so the eval suite and KPI step can
     call the judge without managing an event loop.

The 3-dimension rubric (accuracy / groundedness / format, 1-4 each) is the
W5 rubric, unchanged. Adapt the dimension definitions if your capstone domain
differs; the structure stays put.
"""
from __future__ import annotations

import asyncio
import json
import logging

from pydantic import BaseModel, Field

logger = logging.getLogger(__name__)


# ─── Lazy client (W11 hardening) ─────────────────────────────────────────────
def _async_openai(api_key: str | None = None):
    """Create the AsyncOpenAI client on first use, not at import.

    Importing this module must never require an API key — unit tests import it
    with no environment set. The `openai` import is also deferred so a machine
    without the package can still import the rubric constants.
    """
    from openai import AsyncOpenAI

    return AsyncOpenAI(api_key=api_key)


# ─── Output schema ───────────────────────────────────────────────────────────
class JudgeScore(BaseModel):
    """One judge verdict on one candidate answer."""

    accuracy:     int = Field(..., ge=1, le=4)
    groundedness: int = Field(..., ge=1, le=4)
    format:       int = Field(..., ge=1, le=4)
    reasoning:    str = Field(..., min_length=1)

    @property
    def mean(self) -> float:
        """Average across the three dimensions — useful for sorting/convergence."""
        return (self.accuracy + self.groundedness + self.format) / 3.0

    @property
    def task_success(self) -> bool:
        """KPI #1 rule: a 'successful' answer is accuracy>=3 AND groundedness>=3.

        Format alone never makes or breaks task success — a well-formatted but
        wrong answer is still a failure. Tune this bar in your ADR (D5).
        """
        return self.accuracy >= 3 and self.groundedness >= 3


# ─── Rubric prompt — Enterprise Knowledge Assistant capstone ─────────────────
RUBRIC_PROMPT: str = """\
You are a senior evaluator for an Enterprise Knowledge Assistant — an LLM-backed
service that answers employees' questions about company policies, processes,
and documents.

You rate each candidate answer on three dimensions, each on a 1-4 scale.

ACCURACY — Is the answer factually correct given the ideal answer?
  1 — Wrong. States something that contradicts the ideal answer.
  2 — Partially correct. Some facts right, some wrong or missing critical detail.
  3 — Correct. All stated facts align with the ideal; minor omissions acceptable.
  4 — Correct and complete. Captures every fact in the ideal answer.

GROUNDEDNESS — Does every claim trace to the ideal/source, or does the answer
honestly admit uncertainty when relevant?
  1 — Hallucinates. Invents facts not in the source.
  2 — Mostly grounded but adds 1-2 unsupported claims.
  3 — Grounded. All claims trace to the source. May lack citations.
  4 — Grounded and explicit. Claims trace clearly; cites where possible; admits
      uncertainty if the ideal answer would.

FORMAT — Is the structure, length, and presentation right for the question?
  1 — Wrong shape. Too long/short, missing required structure, garbled.
  2 — Functional but rough. Right info, wrong shape.
  3 — Appropriate. Right length, clear structure, reasonable presentation.
  4 — Polished. Length and structure match what a senior writer would produce.

ALWAYS call the rate_answer tool. Never reply in plain text. The reasoning field
should be 2-4 sentences naming what's strong and what's weak.
"""


# ─── Tool schema for structured output ───────────────────────────────────────
RATE_ANSWER_TOOL: dict = {
    "type": "function",
    "function": {
        "name": "rate_answer",
        "description": (
            "Rate a candidate answer on three dimensions, 1-4 each, "
            "and explain your reasoning."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "accuracy":     {"type": "integer", "minimum": 1, "maximum": 4},
                "groundedness": {"type": "integer", "minimum": 1, "maximum": 4},
                "format":       {"type": "integer", "minimum": 1, "maximum": 4},
                "reasoning":    {"type": "string"},
            },
            "required": ["accuracy", "groundedness", "format", "reasoning"],
        },
    },
}


# ─── Judge call ──────────────────────────────────────────────────────────────
async def judge_one(
    question: str,
    candidate_answer: str,
    ideal_answer: str,
    *,
    judge_model: str = "gpt-4o",
    api_key: str | None = None,
) -> JudgeScore:
    """Run the judge on one (question, candidate, ideal) triple."""
    client = _async_openai(api_key)

    user_msg = (
        f"QUESTION:\n{question}\n\n"
        f"IDEAL ANSWER:\n{ideal_answer}\n\n"
        f"CANDIDATE ANSWER:\n{candidate_answer}\n\n"
        "Rate the candidate using the rate_answer tool."
    )

    resp = await client.chat.completions.create(
        model=judge_model,
        messages=[
            {"role": "system", "content": RUBRIC_PROMPT},
            {"role": "user",   "content": user_msg},
        ],
        tools=[RATE_ANSWER_TOOL],
        tool_choice={"type": "function", "function": {"name": "rate_answer"}},
    )

    tool_calls = resp.choices[0].message.tool_calls or []
    if not tool_calls:
        raise RuntimeError(
            "Judge did not call the rate_answer tool. Check the rubric prompt."
        )
    args = json.loads(tool_calls[0].function.arguments)
    return JudgeScore.model_validate(args)


def judge_one_sync(
    question: str,
    candidate_answer: str,
    ideal_answer: str,
    *,
    judge_model: str = "gpt-4o",
    api_key: str | None = None,
) -> JudgeScore:
    """Synchronous convenience wrapper around `judge_one`.

    The eval suite (test_kpi.py) is synchronous pytest; this lets it grade
    task success without managing an event loop.
    """
    return asyncio.run(
        judge_one(
            question, candidate_answer, ideal_answer,
            judge_model=judge_model, api_key=api_key,
        )
    )


**The task-success rule, exercised locally** (no API call) — KPI #1 counts an answer successful when accuracy ≥ 3 **and** groundedness ≥ 3; format alone never rescues a wrong answer:

In [10]:
good = JudgeScore(accuracy=4, groundedness=3, format=2, reasoning='correct, a bit terse')
bad  = JudgeScore(accuracy=2, groundedness=4, format=4, reasoning='beautifully wrong')
print('good  -> mean', round(good.mean, 2), '| task_success:', good.task_success)
print('bad   -> mean', round(bad.mean, 2),  '| task_success:', bad.task_success)

good  -> mean 3.0 | task_success: True
bad   -> mean 3.33 | task_success: False


## Recap

Four numbers, one pass: **faithfulness** and **answer relevance** read on the *answer*; **context precision** and **context recall** read on *retrieval*. A low score is a **first hypothesis** about which layer to inspect — confirm before you fix (a low answer metric often has a retrieval root cause). Next, **Notebook 2** turns these into the 8-KPI dashboard, persists per-query cost/latency to `results.db`, and sets thresholds you can defend.